In [1]:
import torch
import transformers
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

d:\Programs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.14.0+cu132
Transformers version: 5.17.0
CUDA available: True
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
CUDA version: 13.2


In [2]:
from datasets import load_from_disk

train_dataset = load_from_disk("data/tokenized/train")
validation_dataset = load_from_disk("data/tokenized/validation")
test_dataset = load_from_disk("data/tokenized/test")

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 50000
Validation samples: 10000
Testing samples: 10000


In [3]:
print(f"Dataset columns: {train_dataset.column_names}")
print(f"First training label: {train_dataset[0]['label']}")
print(f"Input IDs length: {len(train_dataset[0]['input_ids'])}")

Dataset columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
First training label: 1
Input IDs length: 192


In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

print(f"Number of Parameters: {model.num_parameters()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 445.61it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Number of Parameters: 109483778


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(f"Model device: {next(model.parameters()).device}")

Model device: cuda:0


In [6]:
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

CUDA device: NVIDIA GeForce GTX 1650 with Max-Q Design
CUDA capability: (7, 5)


In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-amazon-sentiment",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # Optimizer
    learning_rate=2e-5,
    weight_decay=0.01,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    # Keep the best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Performance
    fp16=True,

    # Logging
    logging_strategy="steps",
    logging_steps=100,

    # Checkpoints
    save_total_limit=2,

    # Reproducibility
    seed=42,

    # Disable unnecessary reporting
    report_to="none"
)

print("Training configuration created successfully.")

Training configuration created successfully.


In [8]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    ).numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    auc = roc_auc_score(
        labels,
        probabilities[:, 1]
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [9]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,      
    eval_dataset = validation_dataset,
    compute_metrics = compute_metrics
)

print("Trainer created successfully.")

Trainer created successfully.


In [10]:
torch.cuda.empty_cache()

print(f"GPU memory allocated before test: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"GPU memory reserved before test: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

GPU memory allocated before test: 418.74 MB
GPU memory reserved before test: 472.00 MB


In [11]:
from transformers import TrainingArguments

test_args = TrainingArguments(
    output_dir="./bert-test-run",
    max_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=1,
    report_to="none"
)

In [12]:
test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics
)

In [13]:
test_trainer.train()

Step,Training Loss
1,3.099182


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.66s/it]


TrainOutput(global_step=1, training_loss=3.09918212890625, metrics={'train_runtime': 22.7251, 'train_samples_per_second': 0.704, 'train_steps_per_second': 0.044, 'total_flos': 1578666332160.0, 'train_loss': 3.09918212890625, 'epoch': 0.00032})

In [14]:
del test_trainer

torch.cuda.empty_cache()

print(f"GPU memory allocated after test: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"GPU memory reserved after test: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

GPU memory allocated after test: 1276.04 MB
GPU memory reserved after test: 1606.00 MB


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics
)

print("Final Trainer ready.")

Final Trainer ready.


In [16]:
# train_result = trainer.train()

In [17]:
# Create smaller subsets for practical training on the GTX 1650

train_small = train_dataset.shuffle(seed=42).select(range(10_000))
validation_small = validation_dataset.shuffle(seed=42).select(range(2_000))

print(f"Training samples: {len(train_small)}")
print(f"Validation samples: {len(validation_small)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 10000
Validation samples: 2000
Testing samples: 10000


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-amazon-sentiment",

    num_train_epochs=2,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,

    seed=42,
    report_to="none"
)

print("Training configuration updated successfully.")

Training configuration updated successfully.


In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=validation_small,
    compute_metrics=compute_metrics
)

print("Trainer ready for the smaller training run.")

Trainer ready for the smaller training run.


In [20]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)

print(f"Model reset to pretrained BERT.")
print(f"Model device: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 391.45it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Model reset to pretrained BERT.
Model device: cuda:0


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=validation_small,
    compute_metrics=compute_metrics
)

print("Trainer recreated with corrected metrics.")

Trainer recreated with corrected metrics.


In [22]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.976074,0.218936,0.929000,0.895652,0.964620,0.928858,0.984369
2,0.472990,0.253664,0.940000,0.924319,0.953174,0.938525,0.985509


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.50s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

In [23]:
test_results = trainer.evaluate(
    eval_dataset=test_dataset
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Auc
0.472990,0.195215,2,0.939400,0.920076,0.962400,0.940762,0.985268


{'eval_loss': 0.1952146291732788, 'eval_accuracy': 0.9394, 'eval_precision': 0.920076481835564, 'eval_recall': 0.9624, 'eval_f1': 0.9407624633431085, 'eval_auc': 0.98526768}


In [24]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Get predictions on the test set
predictions = trainer.predict(test_dataset)

# Convert logits to predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

# True labels
y_true = np.array(test_dataset["label"])

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print(f"Confusion Matrix:\n{cm}")

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Negative", "Positive"],
        digits=4
    )
)

Confusion Matrix:
[[4582  418]
 [ 188 4812]]

Classification Report:
              precision    recall  f1-score   support

    Negative     0.9606    0.9164    0.9380      5000
    Positive     0.9201    0.9624    0.9408      5000

    accuracy                         0.9394     10000
   macro avg     0.9403    0.9394    0.9394     10000
weighted avg     0.9403    0.9394    0.9394     10000



In [25]:
# Get test predictions
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = np.array(test_dataset["label"])

# Find misclassified examples
misclassified_indices = np.where(y_pred != y_true)[0]

print(f"Total test samples: {len(y_true)}")
print(f"Misclassified samples: {len(misclassified_indices)}")
print(f"Test accuracy: {(y_pred == y_true).mean():.4f}")

Total test samples: 10000
Misclassified samples: 606
Test accuracy: 0.9394


In [26]:
# Display some misclassified examples

for i in misclassified_indices[:10]:
    print(f"\n{'=' * 80}")
    print(f"True label: {y_true[i]}")
    print(f"Predicted label: {y_pred[i]}")
    print(f"Review: {test_dataset[i]['text']}")


True label: 1
Predicted label: 0


KeyError: 'text'

In [28]:
print(test_dataset.column_names)

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [29]:
print(test_data.column_names)

NameError: name 'test_data' is not defined

In [30]:
%whos

Variable                             Type                             Data/Info
-------------------------------------------------------------------------------
AutoModelForSequenceClassification   type                             <class 'transformers.mode<...>rSequenceClassification'>
AutoTokenizer                        type                             <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
Trainer                              type                             <class 'transformers.trainer.Trainer'>
TrainingArguments                    type                             <class 'transformers.trai<...>_args.TrainingArguments'>
accuracy_score                       function                         <function accuracy_score at 0x000001DF3AE72C40>
classification_report                function                         <function classification_<...>rt at 0x000001DF3AE90250>
cm                                   ndarray                          2x2: 4 elems, type `int64`, 32 bytes
co

In [31]:
from pathlib import Path

for path in Path(
    r"D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer"
).rglob("*"):
    if path.is_file():
        print(path)

D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\01_data_collection_eda.ipynb
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\02_bert_finetuning.ipynb
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\test\data-00000-of-00001.arrow
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\test\dataset_info.json
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\test\state.json
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\train\cache-e053493f5290295b.arrow
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\train\data-00000-of-00001.arrow
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\train\dataset_info.json
D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\data\tokenized\train\state.json
D:\Study\Data S

In [32]:
from datasets import load_dataset, concatenate_datasets

# Load the original Amazon Polarity dataset
raw_dataset = load_dataset("fancyzhx/amazon_polarity")

# Separate positive and negative test reviews
positive_test = raw_dataset["test"].filter(lambda x: x["label"] == 1)
negative_test = raw_dataset["test"].filter(lambda x: x["label"] == 0)

# Recreate the exact 10,000-review test subset used in Notebook 01
test_positive = positive_test.shuffle(seed=42).select(range(5000))
test_negative = negative_test.shuffle(seed=42).select(range(5000))

raw_test_subset = concatenate_datasets(
    [test_positive, test_negative]
).shuffle(seed=42)

# Recreate the combined text exactly as Notebook 01
def combine_text(example):
    title = example["title"].strip()
    content = example["content"].strip()

    if title:
        example["text"] = f"{title}. {content}"
    else:
        example["text"] = content

    return example

raw_test_subset = raw_test_subset.map(combine_text)

print(f"Raw test samples: {len(raw_test_subset)}")
print(f"Columns: {raw_test_subset.column_names}")

Map: 100%|██████████| 10000/10000 [00:00<00:00, 10250.20 examples/s]

Raw test samples: 10000
Columns: ['label', 'title', 'content', 'text']


In [33]:
for i in misclassified_indices[:10]:
    print(f"\n{'=' * 80}")
    print(f"True label: {y_true[i]}")
    print(f"Predicted label: {y_pred[i]}")
    print(f"Title: {raw_test_subset[i]['title']}")
    print(f"Review: {raw_test_subset[i]['content']}")


True label: 1
Predicted label: 0
Title: This animation is not really suitable for children
Review: Great Animation Vol.3Taken fromSpike & Mike's Sick and Twisted Festival of Animation ~ Volume 1(1991)Snookles (1986)Primiti Too Taa (1988)The Thing What Lurked in the Tub (1988)A Story (1987)Feet of Song (1988)Charade (1984)The Hill Farm (1989)Western (1989)Particle Dreams (1988)Winter (1988)Lea Press on Limbs (1988)Bambi Meets Godzilla (1969)

True label: 1
Predicted label: 0
Title: I am a dragon and I agree with this book
Review: I suggest you read this book not all of it is 100% right but yes dragons are from somewhere else but it was not completely all dragon kinds fault that they put sin on mankind. It was actually a higher alien race I call the master, if you want a good read then read this book, I believe is 86% right and the rest like the garden of eden which was later added to the bible is not true. The aliens wrote it up so mankind would see it as some truth and have christians

In [34]:
from collections import Counter

false_positives = np.where((y_true == 0) & (y_pred == 1))[0]
false_negatives = np.where((y_true == 1) & (y_pred == 0))[0]

print(f"Total misclassified: {len(misclassified_indices)}")
print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

Total misclassified: 606
False positives: 418
False negatives: 188


In [35]:
import torch
import numpy as np

# Convert logits to probabilities
logits = predictions.predictions

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=-1
).numpy()

# Confidence = probability assigned to the predicted class
prediction_confidence = probabilities[
    np.arange(len(y_pred)),
    y_pred
]

# Confidence of misclassified predictions
error_confidence = prediction_confidence[misclassified_indices]

print(f"Average confidence on errors: {error_confidence.mean():.4f}")
print(f"Minimum confidence on errors: {error_confidence.min():.4f}")
print(f"Maximum confidence on errors: {error_confidence.max():.4f}")

print(f"\nErrors with confidence >= 90%: {(error_confidence >= 0.90).sum()}")
print(f"Errors with confidence >= 95%: {(error_confidence >= 0.95).sum()}")
print(f"Errors with confidence >= 99%: {(error_confidence >= 0.99).sum()}")

Average confidence on errors: 0.8715
Minimum confidence on errors: 0.5038
Maximum confidence on errors: 0.9982

Errors with confidence >= 90%: 351
Errors with confidence >= 95%: 269
Errors with confidence >= 99%: 106


In [36]:
# Sort misclassified examples by confidence
sorted_error_indices = misclassified_indices[
    np.argsort(error_confidence)[::-1]
]

print("Top 10 highest-confidence errors:")

for i in sorted_error_indices[:10]:
    print(f"\n{'=' * 80}")
    print(f"True label: {y_true[i]}")
    print(f"Predicted label: {y_pred[i]}")
    print(f"Confidence: {prediction_confidence[i]:.4f}")
    print(f"Title: {raw_test_subset[i]['title']}")
    print(f"Review: {raw_test_subset[i]['content']}")

Top 10 highest-confidence errors:

True label: 1
Predicted label: 0
Confidence: 0.9982
Title: WHY CALL IT SEASON 1!
Review: Why call it Season 1 when its only half a season...Season 2 is the second half of Season 1!!! So misleading and when I wrote in about it they told me I should have read the reviews first!!! Why would I read a review if I know what I want to buy??? How about marking the DVD correctly and call it what it is!!! False advertising.

True label: 1
Predicted label: 0
Confidence: 0.9981
Title: Too large
Review: The contents was as expected. The size is much too large. If you mentioned the size, I missed it. Not suitable as a gift at this time.

True label: 1
Predicted label: 0
Confidence: 0.9980
Title: errors on your review page
Review: I am the author. You have listed my name as a reviewer! Please remove it as a reviewer. It looks ridiculous.

True label: 1
Predicted label: 0
Confidence: 0.9977
Title: Disappointed!
Review: Can't believe this GREAT movie, "The Hanging Tre

In [38]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")

Tokenizer loaded: BertTokenizer


In [39]:
tokenizer.save_pretrained(save_path)

print(f"Tokenizer saved to: {save_path}")

Tokenizer saved to: D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\bert-amazon-sentiment\final_model


In [40]:
from pathlib import Path

files = list(Path(save_path).iterdir())

print("Final model files:")

for file in files:
    print(f"- {file.name}")

Final model files:
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


In [41]:
tokenizer.save_pretrained(save_path)

print("Tokenizer saved successfully.")

Tokenizer saved successfully.


In [42]:
from pathlib import Path

for file in sorted(Path(save_path).iterdir()):
    print(f"- {file.name}")

- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


In [43]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    save_path
)

loaded_model.to(device)

print(f"Model loaded successfully on: {device}")
print(f"Number of parameters: {loaded_model.num_parameters():,}")

Loading weights: 100%|██████████| 201/201 [00:03<00:00, 56.98it/s]


Model loaded successfully on: cuda
Number of parameters: 109,483,778


In [44]:
import torch
import torch.nn.functional as F

def predict_sentiment(text):
    loaded_model.eval()

    inputs = loaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=192
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = loaded_model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=-1)[0]

    predicted_class = torch.argmax(probabilities).item()
    confidence = probabilities[predicted_class].item()

    labels = {
        0: "Negative",
        1: "Positive"
    }

    return {
        "sentiment": labels[predicted_class],
        "confidence": confidence,
        "negative_probability": probabilities[0].item(),
        "positive_probability": probabilities[1].item()
    }

In [45]:
review = "This product is absolutely amazing. The quality is excellent and I would definitely buy it again."

result = predict_sentiment(review)

print(f"Review: {review}")
print(f"Sentiment: {result['sentiment']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Negative probability: {result['negative_probability']:.4f}")
print(f"Positive probability: {result['positive_probability']:.4f}")

Review: This product is absolutely amazing. The quality is excellent and I would definitely buy it again.
Sentiment: Positive
Confidence: 0.9964
Negative probability: 0.0036
Positive probability: 0.9964


In [46]:
review = "This product is terrible. It broke after two days and I completely regret buying it."

result = predict_sentiment(review)

print(f"Review: {review}")
print(f"Sentiment: {result['sentiment']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Negative probability: {result['negative_probability']:.4f}")
print(f"Positive probability: {result['positive_probability']:.4f}")

Review: This product is terrible. It broke after two days and I completely regret buying it.
Sentiment: Negative
Confidence: 0.9983
Negative probability: 0.9983
Positive probability: 0.0017


In [47]:
from pathlib import Path
import shutil

project_root = Path(
    r"D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer"
)

source_model = project_root / "notebooks" / "bert-amazon-sentiment" / "final_model"
production_model = project_root / "models" / "amazon-sentiment-bert"

production_model.mkdir(parents=True, exist_ok=True)

print(f"Source: {source_model}")
print(f"Destination: {production_model}")

Source: D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\notebooks\bert-amazon-sentiment\final_model
Destination: D:\Study\Data Science\Claude\Amazon-Review-Sentiment-Analyzer\models\amazon-sentiment-bert


In [48]:
for item in source_model.iterdir():
    destination = production_model / item.name

    if item.is_dir():
        shutil.copytree(item, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(item, destination)

print("Final model copied successfully.")

Final model copied successfully.


In [49]:
print("Production model files:")

for file in sorted(production_model.iterdir()):
    print(f"- {file.name}")

Production model files:
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


In [50]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

production_tokenizer = AutoTokenizer.from_pretrained(
    str(production_model)
)

production_model_loaded = AutoModelForSequenceClassification.from_pretrained(
    str(production_model)
)

production_model_loaded.to(device)
production_model_loaded.eval()

print("Production model loaded successfully.")
print(f"Device: {device}")
print(f"Parameters: {production_model_loaded.num_parameters():,}")

Loading weights: 100%|██████████| 201/201 [00:03<00:00, 53.17it/s]


Production model loaded successfully.
Device: cuda
Parameters: 109,483,778


In [51]:
test_review = "The product is fantastic and the quality is much better than I expected."

inputs = production_tokenizer(
    test_review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=192
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = production_model_loaded(**inputs)

probabilities = torch.softmax(outputs.logits, dim=-1)[0]

predicted_class = torch.argmax(probabilities).item()

labels = {
    0: "Negative",
    1: "Positive"
}

print(f"Review: {test_review}")
print(f"Sentiment: {labels[predicted_class]}")
print(f"Confidence: {probabilities[predicted_class].item():.4f}")

Review: The product is fantastic and the quality is much better than I expected.
Sentiment: Positive
Confidence: 0.9944
